# O*NET preprocessing

Turns the O*NET database into the same seven `processed/*.csv` files the ESCO notebooks write
(same columns, same attribute vocabulary), so `kb_graph.KBGraph.from_processed` works on it
unchanged. `python sources/build_kg_cache.py` builds the graph (both KBs).

Raw files, both official and kept next to this notebook (nothing unpacks the zip):

- `db_31_0_csv.zip`: O*NET 31.0 Database (August 2026), https://www.onetcenter.org/dl_files/database/db_31_0_csv.zip
- `SOC_Structure.csv`: O*NET-SOC 2019 taxonomy, https://www.onetcenter.org/taxonomy/2019/structure/SOC_Structure.csv?fmt=csv
  (the database has no titles for the SOC major / minor / broad groups; this file has them)

The O*NET content is licensed CC BY 4.0.

In [1]:
import sys; sys.path.append("../..")

import uuid
import zipfile

import numpy as np
import pandas as pd

from kb_graph import taxonomy_levels

_ZIP = "db_31_0_csv.zip"
_ZIP_PREFIX = "db_31_0_csv/"


def onet_csv(name: str, **kwargs) -> pd.DataFrame:
    """Read one raw O*NET CSV straight out of the distribution zip (mirror of `esco_csv`)."""
    with zipfile.ZipFile(_ZIP) as archive:
        return pd.read_csv(archive.open(_ZIP_PREFIX + name), **kwargs)

Edge thresholds. Every occupation is rated on every Skill / Knowledge element, so linking them all
would make the graph almost complete bipartite. `IM` is the Importance scale (1-5).

In [2]:
IM_MIN = 3.0        # keep an occupation -> skill/knowledge edge from this importance on
IM_ESSENTIAL = 3.5  # ... and call it `essential` from this one on, `optional` below

# Occupations

`SOC_Structure.csv` lists the taxonomy top-down, one code per row in the column of its level:
Major Group, Minor Group, Broad Occupation, Detailed Occupation, Detailed O*NET-SOC.
Forward-filling the columns down the rows gives, on every row, the code of its current ancestor at
each higher level, so a row's parent is the forward-filled value one column to its left.

The database codes detailed occupations as `XX-XXXX.00` and their O*NET specialisations as
`XX-XXXX.01`, `.02`, ... The structure file writes the former without `.00`, so it is added here:
a specialisation then hangs under its detailed occupation, like ESCO's `2654.1.7` under `2654.1`.
The SOC groups become occupation nodes, like the ISCO groups in ESCO.

In [3]:
LEVELS = ["Major Group", "Minor Group", "Broad Occupation", "Detailed Occupation", "Detailed O*NET-SOC"]
soc = pd.read_csv("SOC_Structure.csv", dtype=str)
assert (soc[LEVELS].notna().sum(axis=1) == 1).all(), "one code per row"

depth = soc[LEVELS].notna().values.argmax(axis=1)          # 0 = major ... 4 = O*NET specialisation
own = soc[LEVELS].ffill(axis=1).iloc[:, -1]                 # the row's own code
ancestors = soc[LEVELS].ffill(axis=0)                       # current ancestor code at every level


def occ_id(code: str, level: int) -> str:
    return code + ".00" if level == 3 else code


soc_nodes = pd.DataFrame({
    "occ_id": [occ_id(c, d) for c, d in zip(own, depth)],
    "label": soc["SOC or O*NET-SOC 2019 Title"],
    "parent": [None if d == 0 else occ_id(ancestors.iat[i, d - 1], d - 1) for i, d in enumerate(depth)],
})
assert soc_nodes.occ_id.is_unique
soc_nodes

,occ_id,label,parent
0,11-0000,Management Occupations,NaN
1,11-1000,Top Executives,11-0000
2,11-1010,Chief Executives,11-1000
3,11-1011.00,Chief Executives,11-1010
4,11-1011.03,Chief Sustainability Officers,11-1011.00
...,...,...,...
1591,55-3014.00,Artillery and Missile Crew Members,55-3010
1592,55-3015.00,Command and Control Center Specialists,55-3010
1593,55-3016.00,Infantry,55-3010
1594,55-3018.00,Special Forces,55-3010


In [4]:
occ_data = onet_csv("occupation_data.csv").set_index("O*NET-SOC Code")
# Every database occupation sits in the taxonomy, and nothing below detailed level is missing from it.
assert set(occ_data.index) == set(soc_nodes.occ_id[depth >= 3])

In [5]:
occ_tax = (soc_nodes.dropna(subset="parent")
           .rename(columns={"occ_id": "conceptUri", "parent": "broaderUri"})[["conceptUri", "broaderUri"]])
assert occ_tax.conceptUri.is_unique, "the occupation taxonomy is a tree: one parent per node"
assert set(occ_tax.broaderUri) <= set(soc_nodes.occ_id)
occ_tax.to_csv("processed/occupation_tax.csv", index=False)

Alternative labels: the O*NET job titles plus the sample of titles reported by incumbents.
They are replaced by integer ids, exactly like the ESCO notebook does.

In [6]:
titles = pd.concat([
    onet_csv("job_titles.csv").rename(columns={"Job Title": "alt"}),
    onet_csv("sample_of_reported_titles.csv").rename(columns={"Reported Job Title": "alt"}),
])[["O*NET-SOC Code", "alt"]].dropna().drop_duplicates()

alt_label2id = {label: i for i, label in enumerate(titles["alt"].drop_duplicates())}
alt_by_occ = titles.groupby("O*NET-SOC Code")["alt"].agg(lambda s: sorted(alt_label2id[a] for a in s))

(pd.Series(alt_label2id).rename_axis("alt_label").rename("alt_label_id").reset_index()
 .to_csv("processed/occ_alt_labels.csv", index=False))

In [7]:
jobs = soc_nodes.set_index("occ_id")[["label"]]
# The database title where there is one: it is the one the O*NET files use.
jobs["label"] = occ_data["Title"].reindex(jobs.index).fillna(jobs["label"])
jobs["altLabels"] = [alt_by_occ.get(i, []) for i in jobs.index]
jobs["code"] = jobs.index

level, leaf = taxonomy_levels(occ_tax)
# Indexing, not .map: an occupation outside the taxonomy must raise here.
jobs["leaf"] = [leaf[i] for i in jobs.index]
jobs["level"] = [level[i] for i in jobs.index]
jobs["description"] = occ_data["Description"].reindex(jobs.index).fillna("")

assert jobs.index.is_unique
jobs.to_csv("processed/occupations.csv")
jobs

,label,altLabels,code,leaf,level,description
occ_id,,,,,,
11-0000,Management Occupations,[],11-0000,False,1,
11-1000,Top Executives,[],11-1000,False,2,
11-1010,Chief Executives,[],11-1010,False,3,
11-1011.00,Chief Executives,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",11-1011.00,False,4,Determine and formulate policies and provide o...
11-1011.03,Chief Sustainability Officers,"[75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 8...",11-1011.03,True,5,"Communicate and coordinate with management, sh..."
...,...,...,...,...,...,...
55-3014.00,Artillery and Missile Crew Members,"[25561, 28277, 44554, 44555, 44556, 44557, 445...",55-3014.00,True,4,"Target, fire, and maintain weapons used to des..."
55-3015.00,Command and Control Center Specialists,"[5027, 6537, 13132, 15813, 15832, 15915, 44612...",55-3015.00,True,4,"Operate and monitor communications, detection,..."
55-3016.00,Infantry,"[44655, 44656, 44657, 44658, 44659, 44660, 446...",55-3016.00,True,4,Operate weapons and equipment in ground combat...


# Skills

Five domains of the O*NET content model, each a tree keyed by its `Element ID`
(the parent of `2.C.4.a` is `2.C.4`):

| root | domain | skill nodes (leaves) |
|---|---|---|
| 2.A | Essential Skills | 10 |
| 2.B | Transferable Skills | 25 |
| 2.C | Knowledge | 33 |
| 4.A | Work Activities | 2087 DWAs, under IWAs and GWAs |
| 2.E | Software Skills | the tool examples ("Python", "Microsoft Excel", ...) under their 134 classes |

Abilities, Work Styles and Work Context are personal traits or conditions, not skills, and are left out.

In [8]:
ROOTS = {"2.A": "Essential Skills", "2.B": "Transferable Skills", "2.C": "Knowledge",
         "2.E": "Software Skills", "4.A": "Work Activities"}

cm = onet_csv("content_model_reference.csv")
cm = cm[cm["Element ID"].str.match(r"(2\.[ABCE]|4\.A)(\.|$)")].set_index("Element ID")
cm["root"] = cm.index.str[:3]

element_tax = pd.DataFrame({"conceptUri": cm.index[~cm.index.isin(list(ROOTS))]})
element_tax["broaderUri"] = element_tax["conceptUri"].str.rsplit(".", n=1).str[0]
assert set(element_tax.broaderUri) <= set(cm.index)

Software tools have no id of their own in O*NET. Each one gets a `uuid5` of its name: stable across runs,
and UUID-shaped like the ESCO skill ids. A tool listed under several classes (151 of them) keeps
one node with several parents, the same kind of DAG as the ESCO skill pillar.

In [9]:
software = onet_csv("software_skills.csv")
software["skill_id"] = [str(uuid.uuid5(uuid.NAMESPACE_URL, "onet:software:" + n)) for n in software["Workplace Example"]]

tool_tax = (software[["skill_id", "Element ID"]].drop_duplicates()
            .rename(columns={"skill_id": "conceptUri", "Element ID": "broaderUri"}))

skill_tax = pd.concat([element_tax, tool_tax], ignore_index=True)
assert not skill_tax.duplicated().any()
skill_tax.to_csv("processed/skill_tax.csv", index=False)

`skill_type` and `reuseLevel` stay inside the ESCO vocabulary, so the filters written for ESCO
(`skill_pool_uris` dropping `broad_skill`, `is_language`) mean the same thing here:

- every content-model element that groups others (domains, GWA, IWA, software classes): `broad_skill` / `group`
- Essential and Transferable Skills: `skill/competence` / `transversal`
- Knowledge: `knowledge` / `cross-sector`; "English Language" and "Foreign Language": `language`
- DWAs and software tools: `skill/competence` / `sector-specific`

In [10]:
has_children = set(element_tax.broaderUri)
elements = pd.DataFrame({"label": cm["Element Name"], "root": cm["root"]})
# reuseLevel by domain.
is_group = elements.index.isin(has_children) | (elements.root == "2.E")
elements["skill_type"] = np.select(
    [is_group, elements.root == "2.C"], ["broad_skill", "knowledge"], "skill/competence")
elements["reuseLevel"] = np.select(
    [is_group, elements.root.isin(["2.A", "2.B"]), elements.root == "2.C"],
    ["group", "transversal", "cross-sector"], "sector-specific")
elements.loc[["2.C.7.a", "2.C.7.b"], "skill_type"] = "language"

tools = (software.groupby("skill_id")
         .agg(label=("Workplace Example", "first"))
         .assign(root="2.E", skill_type="skill/competence", reuseLevel="sector-specific"))

skills = pd.concat([elements, tools]).rename_axis("skill_id")

The pipeline maps skills by label, so the mappable labels (every non-`broad_skill` node) must be
unique. Only "Mathematics" collides (an Essential Skill and a Knowledge element): colliding
labels get their domain as a suffix. A group can still share its name with one of its
members (an IWA and a DWA); `get_uri(..., reduce="Down")` resolves that to the member.

In [11]:
pool = skills.skill_type != "broad_skill"
clash = pool & skills.label.isin(skills.label[pool][skills.label[pool].duplicated()])
skills.loc[clash, "label"] = skills.loc[clash, "label"] + " (" + skills.loc[clash, "root"].map(ROOTS) + ")"
print(skills.loc[clash, "label"].tolist())
assert skills.loc[pool, "label"].is_unique

['Mathematics (Essential Skills)', 'Mathematics (Knowledge)']


In [12]:
# O*NET has no skill synonyms: the file exists so the loaders find the same seven files as for ESCO.
pd.DataFrame(columns=["alt_label", "alt_label_id"]).to_csv("processed/skills_alt_labels.csv", index=False)
skills["altLabels"] = [[] for _ in range(len(skills))]

level, leaf = taxonomy_levels(skill_tax)
skills["level"] = [level[i] for i in skills.index]
skills["leaf"] = [leaf[i] for i in skills.index]
assert skills.loc[pool, "leaf"].all(), "every mappable skill is a leaf"

# IWAs and DWAs have no description in O*NET: they take their parent's; a tool takes its class(es).
parent_name = element_tax.set_index("conceptUri")["broaderUri"].map(cm["Element Name"])
class_names = software.groupby("skill_id")["Element Name"].agg(lambda s: "; ".join(sorted(set(s))))
skills["description"] = (cm["Description"].reindex(skills.index)
                         .fillna(parent_name.reindex(skills.index))
                         .fillna(skills["label"] + " — " + class_names.reindex(skills.index)))

skills = skills[["skill_type", "label", "reuseLevel", "altLabels", "level", "leaf", "description"]]
assert skills.index.is_unique
skills.to_csv("processed/skills.csv")
skills.skill_type.value_counts()

skill_type
skill/competence    10875
broad_skill           557
knowledge              31
language                2
Name: count, dtype: int64

# Skill - Occupation

- Essential Skills, Transferable Skills, Knowledge: the Importance rating, without the ratings O*NET
  recommends suppressing. An edge from `IM_MIN` on; `essential` from `IM_ESSENTIAL` on.
- DWAs: an occupation reaches a DWA through its tasks. `essential` when at least one of those tasks
  is a Core task, `optional` otherwise.
- Software tools: `essential` when O*NET flags the tool as In Demand for that occupation.

In [13]:
ratings = pd.concat([onet_csv(f) for f in ("essential_skills.csv", "transferable_skills.csv", "knowledge.csv")])
ratings = ratings[(ratings["Scale ID"] == "IM") & (ratings["Recommend Suppress"] != "Y")
                  & (ratings["Data Value"] >= IM_MIN)]
rated_rel = pd.DataFrame({
    "occupationUri": ratings["O*NET-SOC Code"],
    "relationType": np.where(ratings["Data Value"] >= IM_ESSENTIAL, "essential", "optional"),
    "skillUri": ratings["Element ID"],
})

In [14]:
task_type = onet_csv("task_statements.csv")[["O*NET-SOC Code", "Task ID", "Task Type"]]
dwa = onet_csv("tasks_to_dwas.csv").merge(task_type, on=["O*NET-SOC Code", "Task ID"], how="left", validate="many_to_one")
dwa_rel = (dwa.assign(core=dwa["Task Type"].eq("Core"))
           .groupby(["O*NET-SOC Code", "DWA Element ID"])["core"].any().reset_index())
dwa_rel = pd.DataFrame({
    "occupationUri": dwa_rel["O*NET-SOC Code"],
    "relationType": np.where(dwa_rel["core"], "essential", "optional"),
    "skillUri": dwa_rel["DWA Element ID"],
})

In [15]:
tool_rel = (software.assign(in_demand=software["In Demand"].eq("Y"))
            .groupby(["O*NET-SOC Code", "skill_id"])["in_demand"].any().reset_index())
tool_rel = pd.DataFrame({
    "occupationUri": tool_rel["O*NET-SOC Code"],
    "relationType": np.where(tool_rel["in_demand"], "essential", "optional"),
    "skillUri": tool_rel["skill_id"],
})

In [16]:
occskill_rel = pd.concat([rated_rel, dwa_rel, tool_rel], ignore_index=True)

assert not occskill_rel.duplicated(subset=["occupationUri", "skillUri"]).any()
assert set(occskill_rel.relationType) == {"essential", "optional"}
assert set(occskill_rel.occupationUri) <= set(jobs.index)
assert set(occskill_rel.skillUri) <= set(skills.index)
assert (skills.loc[occskill_rel.skillUri, "skill_type"] != "broad_skill").all(), "occupations link leaf skills only"

occskill_rel.to_csv("processed/occ_skill_rel.csv", index=False)
occskill_rel.relationType.value_counts()

relationType
optional     44315
essential    25939
Name: count, dtype: int64